# Movie Recommender System
### Content-Based Movie Recommendations using TMDB Top 10K Movies Dataset

## Project Overview
This project builds a **content-based movie recommendation system**. Given a movie title, the system recommends other movies that are similar based on their textual description (overview) and genre.

## Approach
1. Load and clean a dataset of the top 10,000 TMDB movies
2. Combine each movie's overview and genre into a single "tags" field
3. Convert the tags into numerical vectors using **CountVectorizer** (bag-of-words)
4. Compute pairwise **cosine similarity** between all movie vectors
5. For any given movie, recommend the top 5 most similar movies based on cosine similarity
6. Save the processed data and similarity matrix with `pickle` for use in a Streamlit web app (`app.py`)

## Output
The final deliverable is a Streamlit app where a user selects a movie from a dropdown and receives 5 recommended movies, complete with poster images fetched from the TMDB API.

---
## Step 1: Load the Data
We start by importing pandas and loading the raw TMDB movies dataset.

In [33]:
import pandas as pd

In [34]:
movies = pd.read_csv('/Users/ameerjordan/Desktop/Data/Data Science Portfolio Projects/NETFLIX Movie Recommendation System/top10K-TMDB-movies.csv')

Take a look at the first 10 rows to get a sense of the data's structure and columns.

In [35]:
movies.head(10)

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811
5,667257,Impossible Things,"Family,Drama",es,"Matilde is a woman who, after the death of her...",14.358,2021-06-17,8.6,255
6,129,Spirited Away,"Animation,Family,Fantasy",ja,"A young girl, Chihiro, becomes trapped in a st...",92.056,2001-07-20,8.5,13093
7,730154,Your Eyes Tell,"Romance,Drama",ja,"A tragic accident lead to Kaori's blindness, b...",51.345,2020-10-23,8.5,339
8,372754,Dou kyu sei – Classmates,"Romance,Animation",ja,"Rihito Sajo, an honor student with a perfect s...",14.285,2016-02-20,8.5,239
9,372058,Your Name.,"Romance,Animation,Drama",ja,High schoolers Mitsuha and Taki are complete s...,158.270,2016-08-26,8.5,8895


Check summary statistics for the numeric columns (e.g., `id`, `vote_average`, `vote_count`, `popularity`).

In [36]:
movies.describe()

,id,popularity,vote_average,vote_count
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,161243.505000,34.697267,6.621150,1547.309400
std,211422.046043,211.684175,0.766231,2648.295789
min,5.000000,0.600000,4.600000,200.000000
25%,10127.750000,9.154750,6.100000,315.000000
50%,30002.500000,13.637500,6.600000,583.500000
75%,310133.500000,25.651250,7.200000,1460.000000
max,934761.000000,10436.917000,8.700000,31917.000000


Inspect data types and non-null counts for each column.

In [37]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10000 non-null  int64  
 1   title              10000 non-null  object 
 2   genre              9997 non-null   object 
 3   original_language  10000 non-null  object 
 4   overview           9987 non-null   object 
 5   popularity         10000 non-null  float64
 6   release_date       10000 non-null  object 
 7   vote_average       10000 non-null  float64
 8   vote_count         10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB


Check for any missing values across columns, since missing `overview` or `genre` text could cause issues when building our 'tags' feature.

In [38]:
movies.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

---
## Step 2: Feature Selection
For this recommendation system, we only need a few key columns:
- `id`: the TMDB movie ID (used later to fetch posters via the API)
- `title`: the movie's title (shown to the user)
- `overview`: a text description of the plot
- `genre`: the movie's genre(s)

We'll combine `overview` and `genre` into a single `tags` column, which will serve as the text input for our similarity model.

# Feature Selection Part

First, let's see all available columns in the dataset.

In [39]:
movies.columns

Index(['id', 'title', 'genre', 'original_language', 'overview', 'popularity',
       'release_date', 'vote_average', 'vote_count'],
      dtype='object')

Keep only the columns we need: `id`, `title`, `overview`, and `genre`.

In [40]:
movies = movies[['id', 'title', 'overview', 'genre']]

In [41]:
movies

,id,title,overview,genre
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,"Drama,Crime"
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance"
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime"
3,424,Schindler's List,The true story of how businessman Oskar Schind...,"Drama,History,War"
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...,"Drama,Crime"
...,...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo...","Action,Adventure,Fantasy"
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...,"Action,TV Movie,Science Fiction,Comedy,Adventure"
9997,13995,Captain America,"During World War II, a brave, patriotic Americ...","Action,Science Fiction,War"
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...,"Adventure,Fantasy,Action,Drama"


Create a new `tags` column by concatenating the `overview` and `genre` text together. This combined text captures both the plot and the genre, giving our similarity model more context about each movie.

In [42]:
movies['tags'] = movies['overview'] + movies['genre']

In [43]:
movies

,id,title,overview,genre,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,"Drama,Crime",Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime","Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...,"Drama,History,War",The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...,"Drama,Crime",In the continuing saga of the Corleone crime f...
...,...,...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo...","Action,Adventure,Fantasy","The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...,"Action,TV Movie,Science Fiction,Comedy,Adventure",The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"During World War II, a brave, patriotic Americ...","Action,Science Fiction,War","During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...,"Adventure,Fantasy,Action,Drama",A man named Farmer sets out to rescue his kidn...


Now drop the original `overview` and `genre` columns, since their information has been merged into `tags`. The resulting dataframe (`new_data`) is what we'll use going forward.

In [44]:
new_data = movies.drop(columns = ['overview', 'genre'])

In [45]:
new_data

,id,title,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...
...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...


---
## Step 3: Text Vectorization with CountVectorizer
To compare movies based on their text, we first need to convert the `tags` text into numerical vectors. We'll use **CountVectorizer**, which represents each movie's tags as a "bag of words" — a vector counting how often each word (from a vocabulary of the 10,000 most frequent words, excluding common English stop words) appears.

In [46]:
from sklearn.feature_extraction.text import CountVectorizer

Initialize the vectorizer, limiting the vocabulary to the 10,000 most frequent terms and removing common English stop words (e.g., 'the', 'and', 'is').

In [47]:
cv = CountVectorizer(max_features = 10000, stop_words = 'english')

In [48]:
cv

,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",10000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"


Fit the vectorizer on the `tags` column and transform the text into a matrix of word counts. Each row corresponds to a movie, and each column corresponds to a word in the vocabulary.

In [49]:
vector = cv.fit_transform(new_data['tags'].values.astype('U')).toarray()

Check the shape of the resulting vector matrix — it should be (number of movies, number of vocabulary terms).

In [50]:
vector.shape

(10000, 10000)

---
## Step 4: Compute Cosine Similarity
With each movie represented as a vector, we can measure how similar two movies are by computing the **cosine similarity** between their vectors. A cosine similarity close to 1 means the movies have very similar tags (and are likely similar movies), while a value close to 0 means they're dissimilar.

In [51]:
from sklearn.metrics.pairwise import cosine_similarity

Compute the cosine similarity between every pair of movies. The result is a square matrix where `similarity_matrix[i][j]` represents how similar movie `i` is to movie `j`.

In [52]:
similarity_matrix = cosine_similarity(vector)

In [53]:
similarity_matrix

array([[1.        , 0.05634362, 0.12888482, ..., 0.07559289, 0.11065667,
        0.06388766],
       [0.05634362, 1.        , 0.07624929, ..., 0.        , 0.03636965,
        0.        ],
       [0.12888482, 0.07624929, 1.        , ..., 0.02273314, 0.06655583,
        0.08645856],
       ...,
       [0.07559289, 0.        , 0.02273314, ..., 1.        , 0.03253   ,
        0.02817181],
       [0.11065667, 0.03636965, 0.06655583, ..., 0.03253   , 1.        ,
        0.0412393 ],
       [0.06388766, 0.        , 0.08645856, ..., 0.02817181, 0.0412393 ,
        1.        ]], shape=(10000, 10000))

In [64]:
import numpy as np
import pickle

TOP_K = 15  # plenty more than the 5 you display

top_k_indices = np.argsort(-similarity_matrix, axis=1)[:, :TOP_K+1]  # +1 to include self
top_k_scores = np.take_along_axis(similarity_matrix, top_k_indices, axis=1)

pickle.dump({"indices": top_k_indices, "scores": top_k_scores}, open("similarity_topk.pkl", "wb"))

---
## Step 5: Testing the Recommendation Logic
Before building the final `recommend()` function, let's test the core logic step by step:
1. Find the index of a movie by its title
2. Look up its similarity scores against all other movies
3. Sort the scores in descending order
4. Return the titles of the top similar movies (excluding the movie itself)

Find the row index of a specific movie ("The Godfather") in `new_data`. We use `.item()` to convert the numpy integer to a plain Python int.

In [54]:
new_data[new_data['title'] == "The Godfather"].index[0].item()

2

As a quick test, retrieve the similarity scores for the movie at index 2, sort them in descending order, and print the titles of the top 5 most similar movies.

In [55]:
distance = sorted(list(enumerate(similarity_matrix[2])), reverse=True, key = lambda vector:vector[1])
for i in distance[0:5]:
    print(new_data.iloc[i[0]].title)

The Godfather
The Godfather: Part II
Blood Ties
Joker
Bomb City


---
## Step 6: Build the Recommendation Function
Now we wrap the logic above into a reusable `recommend()` function that takes a movie title and prints the top 5 most similar movies.

In [56]:
def recommend(movies):
    index = new_data[new_data['title'] == movies].index[0]
    distance = sorted(list(enumerate(similarity_matrix[index])), reverse=True, key = lambda vector:vector[1])
    for i in distance[0:5]:
        print(new_data.iloc[i[0]].title)

Test the function with an example movie.

In [57]:
recommend("Iron Man")

Iron Man
Iron Man 3
Guardians of the Galaxy Vol. 2
Avengers: Age of Ultron
Star Wars: Episode III - Revenge of the Sith


---
## Step 7: Save Processed Data for the Streamlit App
Finally, we save the processed movie data (`new_data`) and the cosine similarity matrix (`similarity_matrix`) to disk using `pickle`. These files (`movies_list.pkl` and `similarity.pkl`) are loaded directly by `app.py`, our Streamlit web app, so we don't need to recompute everything every time the app runs.

In [58]:
import pickle

Save the processed movie dataframe (`new_data`) to `movies_list.pkl`.

In [59]:
pickle.dump(new_data, open('movies_list.pkl', 'wb'))

Save the cosine similarity matrix to `similarity.pkl`.

In [60]:
pickle.dump(similarity_matrix, open('similarity.pkl', 'wb'))

Quick sanity check: load `movies_list.pkl` back and confirm it matches the data we saved.

In [62]:
pickle.load(open('movies_list.pkl', 'rb'))

,id,title,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...
...,...,...,...
9995,10196,The Last Airbender,"The story follows the adventures of Aang, a yo..."
9996,331446,Sharknado 3: Oh Hell No!,The sharks take bite out of the East Coast whe...
9997,13995,Captain America,"During World War II, a brave, patriotic Americ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,A man named Farmer sets out to rescue his kidn...


---
## Conclusion
We've built a content-based movie recommender that:
- Cleans and preprocesses the TMDB Top 10K movies dataset
- Converts movie descriptions and genres into numerical vectors via CountVectorizer
- Computes cosine similarity between all movies
- Recommends the top 5 most similar movies for any given title
- Saves the processed data so it can be reused in a Streamlit app (`app.py`)
